# OpenAlex analysis

In [2]:
import pandas as pd

In [3]:
# AltairSaver = altair_save_utils.AltairSaver()

In [4]:
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu

import utils
import importlib
importlib.reload(utils);

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

2024-07-02 14:00:48,309 - botocore.credentials - INFO - Found credentials in environment variables.
2024-07-02 14:00:50,510 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [6]:
# Labelled data
data_df = utils.load_openalex_data().query("topics != 'arts'")

In [64]:
len(data_df.drop_duplicates("id"))

70128

In [15]:
check_words = ['economic']
# check articles that contain the words in check_words and ignore case
check_df = data_df[data_df['text'].str.contains('|'.join(check_words), case=False)]

In [17]:
ts_df = check_df.groupby("year").agg(counts = ('year', 'count')).reset_index()

In [18]:
ts_df

,year,counts
0,2013.0,203
1,2014.0,247
2,2015.0,251
3,2016.0,343
4,2017.0,546
5,2018.0,462
6,2019.0,563
7,2020.0,664
8,2021.0,604
9,2022.0,561


In [497]:
# data_df.topics.value_counts().head(20)

In [20]:
len(data_df.query("year >=2017"))

57810

In [21]:
len(data_df)

70128

In [22]:
# Taxonomy dataframe
topics_df = utils.load_topic_data()

In [23]:
# Transform to one id and topic pair per row
data_exploded_df = utils.explode_data(data_df).query("topic != 'arts'")

## Baseline trends

Baseline trends for publication counts

In [194]:
baseline_df = utils.get_baseline_openalex()

In [195]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = baseline_df,
    year_start = 2019,
    year_end = 2023  
)
trends_baseline

,magnitude,growth
counts,10069983.8,-3.699963


In [196]:
fig = pu.ts_smooth(
    baseline_df.assign(Total="Total").assign(counts = lambda df: df.counts/1e+6),
    ["Total"],
    variable= "counts",
    variable_title = "Publications (millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

## Insight 0: Overall trends

Early-years project growth of funding and project counts trends



In [197]:
ts_counts = utils.get_timeseries(data_df, column='id')

In [198]:
ts_counts

,year,counts
0,2013,2685
1,2014,2958
2,2015,3104
3,2016,3507
4,2017,7579
5,2018,8279
6,2019,8662
7,2020,8572
8,2021,8144
9,2022,7292


In [199]:
au.ts_magnitude_growth_(
    ts_df = ts_counts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,8327.2,-0.48124


In [200]:
fig = pu.ts_smooth(
    ts_counts.assign(Total="Total").query("year >= 2017"),
    ["Total"],
    variable= "counts",
    variable_title = "Publications",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

Funding for the overall early-years development related research has increased by about 19% in the past five years, which is a positive trend compared to baseline funding which slightly decreased by about 5% in the same time period.

In [214]:
utils.get_data_distribution(data_exploded_df.query("year >= 2019"), column='type', values=['id'])

,type,counts,counts_prop
0,Biosciences,4330,0.104
1,Child care & preschool,6065,0.146
2,Development & learning,11692,0.281
3,General,23618,0.567
4,Health,20278,0.487
5,Parenting,1353,0.032
6,Social,9569,0.23
7,Technology,2493,0.06


In [202]:
ts_df = (
    utils.get_data_distribution(data_exploded_df, column='type', values=['id'], ts=True)
    .query("type != 'General'")
)
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='type', value='id')

,magnitude,growth,type,counts
7,498.6,77.950311,Technology,2493
1,1213.0,27.176781,Child care & preschool,6065
6,1913.8,20.922059,Social,9569
5,270.6,17.482517,Parenting,1353
2,2338.4,15.561679,Development & learning,11692
4,4055.6,-3.248007,Health,20278
3,4723.6,-10.857257,General,23618
0,866.0,-16.969274,Biosciences,4330


In [203]:
fig = pu.ts_smooth(
    ts_df,
    ts_df['type'].unique(),
    variable= "counts",
    variable_title = "",
    category_column = 'type',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [357]:
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='subtype', value='id').sort_values(['type', 'growth'], ascending=False)


,magnitude,growth,subtype,counts,type
0,208.6,123.214286,Internet,1043,Technology
1,125.2,91.266376,AI,626,Technology
2,85.2,77.456647,Immersive tech,426,Technology
5,146.4,35.362319,Mobile,732,Technology
3,382.2,46.596244,Inclusion,1911,Society
11,429.6,25.641026,Community,2148,Society
12,728.0,23.685667,Inequalities,3640,Society
15,228.4,20.648464,Labour market,1142,Society
16,781.2,18.365337,Social services,3906,Society
23,264.2,8.381503,Income,1321,Society


## Insight 1: Technology trends

- Magnitude and growth for technology topic overall
- Distribution of different technologies
- Growth of different technologies in UKRI funding


### Overall technology topic growth

In [546]:
tech_subtypes = set(topics_df.query("type == 'Technology'").subtype.unique())
tech_subtypes

{'AI', 'Immersive tech', 'Internet', 'Mobile'}

In [547]:
tech_type_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'type'])
)

In [548]:
# ts_amounts_tech = utils.get_timeseries(tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(tech_type_df, column='id')
utils.plot_quick_ts(ts_counts_tech.query("year >= 2017"), 'counts')

alt.Chart(...)

In [549]:
au.ts_magnitude_growth_(ts_counts_tech, year_start = 2019, year_end = 2023)

,magnitude,growth
counts,498.6,77.950311


In [550]:
498.6*5

2493.0

### Distribution of different technologies

In [209]:
tech_subtype_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    .query("type == 'Technology'")
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'subtype'])
)

In [210]:
# Total tech funding
counts_total = tech_subtype_df.drop_duplicates('id').query("year >= 2019").id.nunique()

In [219]:
tech_subtype_dist = (
    tech_subtype_df
    .query("year >= 2019")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
    .assign(counts_prop = lambda df: df.counts/counts_total)
)

tech_subtype_dist.sort_values('counts_prop', ascending=False)

,subtype,counts,counts_prop
2,Internet,1043,0.418371
3,Mobile,732,0.293622
0,AI,626,0.251103
1,Immersive tech,426,0.170878


### Growth of technology topics

In [221]:
column = 'subtype'
value = 'counts'

tech_subtype_ts = (
    tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

utils.magnitude_and_growth(tech_subtype_ts, column, value).sort_values('growth', ascending=False)

,magnitude,growth,subtype
0,208.6,123.214286,Internet
0,125.2,91.266376,AI
0,85.2,77.456647,Immersive tech
0,146.4,35.362319,Mobile


In [222]:
fig = pu.ts_smooth(
    tech_subtype_ts.query("year >= 2017"),
    ["AI", "Immersive tech", "Internet", "Mobile"],
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 2: Applications

- Where are these technologies applied the most?
- Where do we see growth vs stagnation when it comes to applications?

In [551]:
tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2017")
    .drop_duplicates('id')
    .id.to_list()
)

tech_ids_5y = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2019")
    .drop_duplicates('id')
    .id.to_list()
)

### Application distribution

In [552]:
column = 'type'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology' and type != 'General'"),
    column=column, 
    values=['id'],
    ts=True
)


In [553]:
tech_applications_df.sort_values('counts_prop', ascending=False)

,type,counts,counts_prop
7,Technology,2493,1.0
3,General,1306,0.524
4,Health,906,0.363
2,Development & learning,723,0.29
6,Society,429,0.172
1,Child care & preschool,398,0.16
0,Biosciences,185,0.074
5,Parenting,102,0.041


In [554]:
tech_applications_df.type.to_list()

['Biosciences',
 'Child care & preschool',
 'Development & learning',
 'General',
 'Health',
 'Parenting',
 'Society',
 'Technology']

In [555]:
importlib.reload(pu);
import altair as alt
new_column = 'Category'
ts_df = tech_applications_ts.query("year >= 2017").rename(columns={'type': new_column})

# Define custom colors for each distinct Category
custom_colors = {
    'Biosciences': '#9A1BBE',
    'Child care & preschool': '#EB003B' ,
    'Development & learning': '#FDB633',
    'Health': '#0000FF',
    'Parenting': '#A59AEE',
    'Society': '#18A48C',
}

# Create a color scale using the custom colors
color_scale = alt.Scale(domain=list(custom_colors.keys()), range=list(custom_colors.values()))

fig = pu.ts_smooth(
    ts_df,
    ts_df[new_column].unique(),
    variable= "counts",
    variable_title = "Publication count",
    category_column = new_column,
    width = 400,
    height = 250,
    legend_orient='right',
    color_scale=color_scale  # Pass the custom color scale
)
pu.configure_plots(fig)


alt.Chart(...)

In [556]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id').sort_values('growth', ascending=False)

,magnitude,growth,type,counts
5,20.4,97.297297,Parenting,102
1,79.6,96.078431,Child care & preschool,398
2,144.6,90.405904,Development & learning,723
7,498.6,77.950311,Technology,2493
6,85.8,70.930233,Society,429
0,37.0,65.384615,Biosciences,185
4,181.2,59.349593,Health,906
3,261.2,55.675676,General,1306


In [557]:
trends_df = utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')
(
    tech_applications_df
    .merge(trends_df.drop('counts', axis=1), on='type')[['type', 'magnitude', 'growth', 'counts', 'counts_prop']]
    .query("type != 'Technology' and type != 'General'")
)

,type,magnitude,growth,counts,counts_prop
0,Biosciences,37.0,65.384615,185,0.074
1,Child care & preschool,79.6,96.078431,398,0.16
2,Development & learning,144.6,90.405904,723,0.29
4,Health,181.2,59.349593,906,0.363
5,Parenting,20.4,97.297297,102,0.041
6,Society,85.8,70.930233,429,0.172


In [559]:
from discovery_child_development.utils import chart_trends
importlib.reload(chart_trends);

chart_trends.estimate_trend_type(
    trends_df.query("type != 'Technology' and type != 'General'"), 
    magnitude_column='magnitude', 
    growth_column='growth'
)

,magnitude,growth,type,counts,trend_type_suggestion
5,20.4,97.297297,Parenting,102,emerging
1,79.6,96.078431,Child care & preschool,398,emerging*
2,144.6,90.405904,Development & learning,723,hot
6,85.8,70.930233,Society,429,hot*
0,37.0,65.384615,Biosciences,185,emerging
4,181.2,59.349593,Health,906,hot


In [560]:
# scatter chart of trends_df
import altair as alt
alt.Chart(
    trends_df.query("type != 'Technology' and type != 'General'")
).mark_point().encode(
    x='magnitude:Q',
    y='growth:Q',
    color='type:N',
    tooltip=['type', 'magnitude', 'growth']
)


alt.Chart(...)

In [530]:
tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    # .query("subtype == 'AI'")
    .query("year >= 2017")
    .drop_duplicates('id')
    .id.to_list()
)

In [518]:
# pd.set_option('display.max_colwidth', 200)
# (
#     data_exploded_df
#     .query('id in @tech_ids')
#     .query("type == 'Child care & preschool'")
#     # .query("type == 'Social'")
#     .drop_duplicates(['id'])
#     .sort_values('year', ascending=False)
# )[['id', 'text', 'topics', 'year']]

### Application distribution: More granular subtypes

In [531]:
column = 'subtype'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id'],
    ts=True
)
tech_applications_df.query("type != 'Technology'").sort_values('counts', ascending=False)

,subtype,counts,counts_prop,type
12,Infancy,720,0.289,General
7,Health,538,0.216,Health
5,Games,394,0.158,General
29,Preschool,359,0.144,Child care & preschool
2,Cognitive development,253,0.101,Development & learning
3,Communication and language,203,0.081,Development & learning
28,Prenatal,200,0.08,Health
32,Social services,183,0.073,Society
25,Personal social emotional,173,0.069,Development & learning
20,Non-tech assessments,167,0.067,General


In [532]:
subtypes_trends_df = utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id').sort_values(['type', 'growth'], ascending=False)
subtypes_trends_df

,magnitude,growth,subtype,counts,type
5,208.6,123.214286,Internet,1043,Technology
11,125.2,91.266376,AI,626,Technology
14,85.2,77.456647,Immersive tech,426,Technology
30,146.4,35.362319,Mobile,732,Technology
1,7.0,160.000000,Labour market,35,Society
7,5.4,100.000000,Income,27,Society
13,36.6,84.285714,Social services,183,Society
21,16.4,67.647059,Inclusion,82,Society
27,31.4,45.833333,Inequalities,157,Society
28,23.4,43.636364,Community,117,Society


In [533]:
cat_type = 'Development & learning'
cats = list(topics_df.query("type == @cat_type").subtype.unique())

In [534]:
fig = pu.ts_smooth(
    tech_applications_ts.query("year >= 2017"),
    cats,
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

In [303]:
from discovery_child_development.utils import chart_trends
importlib.reload(chart_trends);

In [535]:
_trends_df = (
    subtypes_trends_df.query("type == 'Development & learning'")
    .rename(columns={'magnitude': 'Magnitude', 'subtype': 'Category'})
    .assign(growth = lambda df: df.growth/100)
)
mid_point = _trends_df.Magnitude.median()
_trends_df

,Magnitude,growth,Category,counts,type
2,34.6,1.415094,Personal social emotional,173,Development & learning
4,50.6,1.273810,Cognitive development,253,Development & learning
12,27.4,0.867925,Mathematics,137,Development & learning
17,40.6,0.697674,Communication and language,203,Development & learning
19,28.6,0.690909,Special educational needs,143,Development & learning
26,16.8,0.512821,Literacy,84,Development & learning


In [536]:
chart_trends.estimate_trend_type(_trends_df)

,Magnitude,growth,Category,counts,type,trend_type_suggestion
2,34.6,1.415094,Personal social emotional,173,Development & learning,hot*
4,50.6,1.273810,Cognitive development,253,Development & learning,hot
12,27.4,0.867925,Mathematics,137,Development & learning,emerging
17,40.6,0.697674,Communication and language,203,Development & learning,hot
19,28.6,0.690909,Special educational needs,143,Development & learning,emerging*
26,16.8,0.512821,Literacy,84,Development & learning,emerging


In [537]:
_trends_df

# make a bubble chart


,Magnitude,growth,Category,counts,type
2,34.6,1.415094,Personal social emotional,173,Development & learning
4,50.6,1.273810,Cognitive development,253,Development & learning
12,27.4,0.867925,Mathematics,137,Development & learning
17,40.6,0.697674,Communication and language,203,Development & learning
19,28.6,0.690909,Special educational needs,143,Development & learning
26,16.8,0.512821,Literacy,84,Development & learning


In [540]:
import altair as alt
colour_field = "type"
text_field = "Category"
height = 250

data = _trends_df

# Chart
fig = (
    alt.Chart(data, width=500, height=height)
    .mark_circle(color=pu.NESTA_COLOURS[0], opacity=1)
    .encode(
        x=alt.X(
            "growth:Q",
            axis=alt.Axis(
                format="%",
                title="Growth",
                labelAlign="center",
                labelExpr="datum.value < -1 ? null : datum.label",
                labelFlush=False,
            ),
            scale=alt.Scale(
                domain=(-0.1, 1.5),
                # domain=(.1, 100), type="log",                
            ),
        ),
        y=alt.Y("Category:N", sort=data["Category"].to_list(), axis=None),
        size=alt.Size(
            "counts",
            title="Publication counts",
            legend=alt.Legend(orient="left"),
            scale=alt.Scale(domain=[0.1, 70]),
        ),
        # color=alt.Color(colour_field, legend=alt.Legend(orient="left")),
        # specify one specific colour
        color = alt.value('#FDB633'),
        # tooltip=[
        #     alt.Tooltip("Category:N", title="Category"),
        #     alt.Tooltip(
        #         "Magnitude:Q",
        #         format=",.3f",
        #         title="Average yearly investment (billion GBP)",
        #     ),
        #     "Number of companies",
        #     "Number of deals",
        #     alt.Tooltip("growth:Q", format=",.0%", title="Growth"),
        # ],
    )
)

# Text labels
text = (
    alt.Chart(data)
    .mark_text(align="left", baseline="middle", font=pu.FONT, dx=20, fontSize=14)
    .encode(
        text=text_field,
        x="growth:Q",
        y=alt.Y("Category:N", sort=data["Category"].to_list(), title=""),
    )
)

# Baseline
baseline_rule = (
    alt.Chart(pd.DataFrame({"x": [-0.035]}))
    .mark_rule(strokeDash=[5, 7], size=1, color="k")
    .encode(x=alt.X("x:Q"))
)

final_fig = pu.configure_titles(pu.configure_axes((baseline_rule + fig + text)), "", "")
final_fig

alt.LayerChart(...)

In [ ]:
colour_field = "subtypes_trends_df"
text_field = "Category"
height = 250

data = _trends_df

In [ ]:
import altair as alt
colour_field = "type"
text_field = "Category"
height = 250

data = _trends_df

# Chart
fig = (
    alt.Chart(data, width=500, height=height)
    .mark_circle(color=pu.NESTA_COLOURS[0], opacity=1)
    .encode(
        x=alt.X(
            "growth:Q",
            axis=alt.Axis(
                format="%",
                title="Growth",
                labelAlign="center",
                labelExpr="datum.value < -1 ? null : datum.label",
                labelFlush=False,
            ),
            scale=alt.Scale(
                domain=(-0.1, 1.5),
                # domain=(.1, 100), type="log",                
            ),
        ),
        y=alt.Y("Category:N", sort=data["Category"].to_list(), axis=None),
        size=alt.Size(
            "counts",
            title="Publication counts",
            legend=alt.Legend(orient="left"),
            scale=alt.Scale(domain=[0.1, 70]),
        ),
        # color=alt.Color(colour_field, legend=alt.Legend(orient="left")),
        # specify one specific colour
        color = alt.value(pu.NESTA_COLOURS[2]),
        # tooltip=[
        #     alt.Tooltip("Category:N", title="Category"),
        #     alt.Tooltip(
        #         "Magnitude:Q",
        #         format=",.3f",
        #         title="Average yearly investment (billion GBP)",
        #     ),
        #     "Number of companies",
        #     "Number of deals",
        #     alt.Tooltip("growth:Q", format=",.0%", title="Growth"),
        # ],
    )
)

# Text labels
text = (
    alt.Chart(data)
    .mark_text(align="left", baseline="middle", font=pu.FONT, dx=20, fontSize=14)
    .encode(
        text=text_field,
        x="growth:Q",
        y=alt.Y("Category:N", sort=data["Category"].to_list(), title=""),
    )
)

# Baseline
baseline_rule = (
    alt.Chart(pd.DataFrame({"x": [-0.035]}))
    .mark_rule(strokeDash=[5, 7], size=1, color="k")
    .encode(x=alt.X("x:Q"))
)

final_fig = pu.configure_titles(pu.configure_axes((baseline_rule + fig + text)), "", "")
final_fig

In [317]:
chart_trends._epsilon = 0.05

fig = chart_trends.mangitude_vs_growth_chart(
    _trends_df,
    x_limit=40,
    y_limit=1.5,
    mid_point=mid_point,
    baseline_growth=0,
    values_label="",
    text_column="Category",
    width=425,
)
fig

alt.LayerChart(...)

### More detailed breakdown of application trends

In [473]:
def get_counts_by_application(selected_ids, groupby_column='name'):
    show_types = ['Biosciences', 'Child care & preschool', 'Development & learning', 'Health', 'Society']
    return (
        data_exploded_df.query("year >= 2019")
        .query("id in @selected_ids")
        .drop_duplicates(['id', 'name'])
        .groupby(groupby_column)
        .agg(counts = ('id', 'count'))
        .reset_index()
        .merge(topics_df, on=groupby_column, how='left')
        .sort_values(['type', 'counts'], ascending=[True, False])
        .query("type in @show_types")
    )[['topic', 'name', 'subtype', 'type', 'counts']]


In [474]:
selected_ids = data_exploded_df.query("type == 'Technology'").id.to_list()
counts_by_application_area_df = get_counts_by_application(selected_ids, groupby_column='name')
counts_by_application_area_df

,topic,name,subtype,type,counts
19,neuroscience,Neuroscience,Neuroscience,Biosciences,130
7,genetics,Genetics,Genetics,Biosciences,64
29,preschool,Preschool,Preschool,Child care & preschool,359
22,operations,Operations,Operations,Child care & preschool,66
3,cognitive,Cognitive development,Cognitive development,Development & learning,253
4,communication,Communication and language,Communication and language,Development & learning,203
25,emotional,Personal social emotional,Personal social emotional,Development & learning,173
35,send,Special educational needs,Special educational needs,Development & learning,143
16,mathematics,Mathematics,Mathematics,Development & learning,137
15,literacy,Literacy,Literacy,Development & learning,84


In [428]:
def get_counts_by_application_chart(
    df,
    chart_title = "Digital technology applications (detailed)",
    chart_subtitle = "Number of publications",
):
    fig = alt.Chart(
        df,
        width=300,
        height=400,
    ).mark_bar().encode(
        y=alt.Y('name:N', sort=df.name.to_list(), title=''),
        x=alt.X('counts:Q', title=''),
        color=alt.Color('type:N', legend=alt.Legend(title='Type')),
        tooltip=['name', 'counts']
    )

    fig = pu.configure_titles(pu.configure_plots(fig), chart_title, chart_subtitle)
    return fig

In [429]:
get_counts_by_application_chart(counts_by_application_area_df)

alt.Chart(...)

In [469]:
counts = []
_df = topics_df.query("type in @show_types").query("topic != 'arts'")[['name']]
for tech_topic in ['AI', 'Internet', 'Mobile', 'Immersive tech']:
    selected_ids = data_exploded_df.query("subtype == @tech_topic").id.to_list()
    counts_df = get_counts_by_application(selected_ids, groupby_column='name')[['counts', 'name']].rename(columns={'counts': tech_topic})
    _df = _df.merge(counts_df, on='name', how='left')
_df = _df.fillna(0)


In [483]:
counts_final_df = counts_by_application_area_df[['name', 'counts']].merge(_df, on='name').rename(columns={'counts': 'Total'})
counts_final_df

,name,Total,AI,Internet,Mobile,Immersive tech
0,Neuroscience,130,71,10,34,33.0
1,Genetics,64,43,8,6,11.0
2,Preschool,359,14,197,105,84.0
3,Operations,66,3,50,13,6.0
4,Cognitive development,253,59,73,103,66.0
5,Communication and language,203,33,77,100,23.0
6,Personal social emotional,173,8,125,42,22.0
7,Special educational needs,143,52,40,42,33.0
8,Mathematics,137,20,45,55,39.0
9,Literacy,84,7,34,41,9.0


In [485]:
counts_final_df.to_csv(utils.PROJECT_DIR / 'outputs/data/tables/tech_applications_openalex.csv', index=False)

In [499]:
importlib.reload(utils);
df = utils.get_counts_by_application(data_exploded_df, topics_df)
df.to_csv(utils.PROJECT_DIR / 'outputs/data/tables/tech_applications_openalex.csv', index=False)
df

,name,Total,AI,Internet,Mobile,Immersive tech
0,Neuroscience,130,71,10,34,33.0
1,Genetics,64,43,8,6,11.0
2,Preschool,359,14,197,105,84.0
3,Operations,66,3,50,13,6.0
4,Cognitive development,253,59,73,103,66.0
5,Communication and language,203,33,77,100,23.0
6,Personal social emotional,173,8,125,42,22.0
7,Special educational needs,143,52,40,42,33.0
8,Mathematics,137,20,45,55,39.0
9,Literacy,84,7,34,41,9.0


## Insight 3: Geographical insights

- Top countries in terms of counts
- UK vs baseline growth for overall counts, in technology counts and application counts

In [561]:
data_countries_df = data_exploded_df.explode('country_code').drop_duplicates(['id', 'country_code'])

n_total = data_countries_df.id.nunique()
n_without_country = data_countries_df.country_code.isnull().sum()

print(f"Number of items without country: {n_without_country} ({n_without_country/n_total:.2%})")

Number of items without country: 12764 (18.30%)


## Overall trends

In [562]:
importlib.reload(utils);

growth_df, ts_counts = utils.get_geographical_distribution(
    data_exploded_df
)

(
    growth_df
    .sort_values('magnitude', ascending=False)
    .head(15)
)

,magnitude,growth,country_code
2,1644.6,-32.545398,US
32,544.8,153.632761,ID
0,438.6,-15.921409,AU
7,381.6,-25.458392,CA
1,371.4,-13.723917,GB
6,323.2,26.778784,CN
11,193.6,9.338521,BR
12,156.0,37.704918,TR
4,148.4,12.590799,DE
10,139.0,-2.307692,SE


In [563]:
countries = ['US', 'GB', 'ID']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "counts",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

### Technology publications

In [564]:
growth_df, ts_counts = utils.get_geographical_distribution(
    data_exploded_df.query('subtype in @tech_subtypes')
)

In [577]:
n_total = data_exploded_df.query('subtype in @tech_subtypes').query("year >= 2019").drop_duplicates('id').id.nunique()

In [578]:
df_ = (
    growth_df
    .sort_values('magnitude', ascending=False)
)

In [580]:
(df_.magnitude*5).sum()/n_total

0.8251103088648215

In [585]:
df_.query("country_code=='US'").magnitude.iloc[0]*5 / n_total

0.17970316887284396

In [584]:
df_.query("country_code=='GB'").magnitude.iloc[0]*5 / n_total

0.036101083032490974

In [588]:
df_.assign(prop = lambda df: df.magnitude*5/n_total).head(10)

,magnitude,growth,country_code,prop
0,89.6,16.956522,US,0.179703
11,61.2,413.043478,ID,0.122744
4,25.4,5.405405,AU,0.050943
12,21.0,45.833333,CN,0.042118
6,18.0,13.461538,GB,0.036101
8,15.8,35.714286,CA,0.031689
17,11.2,122.222222,TR,0.022463
3,8.8,218.181818,IN,0.017649
2,8.2,342.857143,DE,0.016446
41,8.2,16.000000,ES,0.016446


In [566]:
countries = ['US', 'GB']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "counts",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

In [567]:
data_countries_gb_df = (
    data_exploded_df
    .explode('country_code')
    .dropna(subset=['country_code'])
    # .query('subtype in @tech_subtypes')
    .drop_duplicates(['id']) 
    .query("country_code == 'GB'")   
)
len(data_countries_gb_df)

3275

In [568]:
data_countries_gb_df.groupby('subtype').agg(counts=('id', 'nunique')).reset_index()

,subtype,counts
0,AI,12
1,Child protection,88
2,Cognitive development,66
3,Communication and language,79
4,Community,21
5,Games,41
6,Genetics,71
7,Health,222
8,Immersive tech,7
9,Inclusion,20


In [179]:
pd.set_option('display.max_colwidth', 200)
data_countries_gb_df[['id', 'text']]

,id,text
3,W3097547541,Does Household Income Affect children’s Outcomes? A Systematic Review of the Evidence. Abstract There is abundant evidence that children in low income households do less well than their peers on a...
113,W3001856178,Contemporary Outcomes for Infants with Necrotizing Enterocolitis—A Systematic Review. <h3>Objective</h3> To develop an accurate understanding of outcomes for necrotizing enterocolitis (NEC) to inf...
136,W3099266779,Cognitive function in toddlers with congenital heart disease: The impact of a stimulating home environment. Abstract Infants born with congenital heart disease (CHD) are at increased risk of neuro...
143,W3002199400,Does phonetic repertoire in minimally verbal autistic preschoolers predict the severity of later expressive language impairment?. Trajectories of expressive language development are highly heterog...
160,W3011087543,Prediction models for childhood asthma: A systematic review. Abstract Background The inability to objectively diagnose childhood asthma before age five often results in both under‐treatment and ov...
...,...,...
183847,W2411012871,PTH-228 Genotyping of rotavirus isolates prior to the introduction of the rotavirus vaccine in scotland and early indications of the impact of the vaccine. <h3>Introduction</h3> Rotaviruses (RV) a...
184009,W4385239022,"The impact of COVID-19 on routine child immunisation in South Africa. Abstract Background The COVID-19 pandemic disrupted immunisation programs worldwide, reversing gains that had brought vaccine-..."
184033,W4367836025,"Cognitive outcome and its neural correlates after cardiorespiratory arrest in childhood. Abstract Hypoxia-ischaemia (HI) can result in structural brain abnormalities, which in turn can lead to beh..."
184096,W4362700893,"Annual Research Review. ‘There, the dance is - at the still point of the turning world’: dynamic systems perspectives on co-regulation and dysregulation during early development. During developmen..."


## Export data

In [380]:
def check_if_nan_only_list_elements(x):
    return all([pd.isna(i) for i in x])

def convert_topic_columns(df):
    for col in ['topic', 'minor_category', 'major_category', 'topic_code']:
        df = df.assign(**{col : lambda df: df[col].apply(lambda x: list(x) if check_if_nan_only_list_elements(x)==False else [])})
        df = df.assign(**{col : lambda df: df[col].apply(lambda x: ", ".join([xx for xx in x if isinstance(xx, str)]) if len(x)>0 else "")})
    return df

In [396]:
_export_df = (
    data_df
    .fillna({'topics': ''})
    .assign(topics = lambda df: df['topics'].apply(lambda x: [t.strip() for t in x.split(',') if isinstance(t, str)]))
    .explode('topics')
    .merge(topics_df, left_on='topics', right_on='topic', how='left')
    .rename(columns={'topic': 'topic_code', 'type': 'major_category', 'subtype': 'minor_category', 'name': 'topic'})
    .drop('topics', axis=1)
)  

export_df = (
    data_df[['id', 'text', 'dataset', 'year', 'country_code']]
    .merge(
        _export_df[['id', 'topic', 'major_category', 'minor_category', 'topic_code']].groupby('id').agg(set),
        on='id',
        how='left'
    )
    .pipe(convert_topic_columns)
    .assign(url = lambda df: "https://openalex.org/" + df['id'])
    .assign(country_code = lambda df: df.country_code.apply(lambda x: ", ".join([xx for xx in x if isinstance(xx, str)])))
)

In [398]:
export_df.to_csv(utils.PROJECT_DIR / "outputs/data/tables/openalex_final.csv", index=False)

##  Checking "economics" papers

In [24]:
_data_df = utils.load_openalex_data()
_data_exploded_df = utils.explode_data(data_df).query("topic != 'arts'")

In [31]:
topics_df

,topic,type,subtype,name
0,genetics,Biosciences,Genetics,Genetics
1,neuroscience,Biosciences,Neuroscience,Neuroscience
2,operations,Child care & preschool,Operations,Operations
3,preschool,Child care & preschool,Preschool,Preschool
4,cognitive,Development & learning,Cognitive development,Cognitive development
5,communication,Development & learning,Communication and language,Communication and language
6,arts,Development & learning,Expressive arts and design,Expressive arts and design
7,literacy,Development & learning,Literacy,Literacy
8,mathematics,Development & learning,Mathematics,Mathematics
9,emotional,Development & learning,Personal social emotional,Personal social emotional


In [62]:
allowed_topics = ['labour_market', 'social_services', 'inequality', 'income']
_df = (
    _data_exploded_df
    .query("type == 'Society'")
    .query("topic in @allowed_topics")
    .drop_duplicates(['id'])
    .groupby("year")
    .agg(counts=('id', 'nunique'))
    .reset_index()
    .query("year >= 2017")
)

import altair as alt
import discovery_child_development.utils.plotting_utils as pu

# bar chart showing years and a thick line
fig = alt.Chart(_df, width=300).mark_line(
    color=pu.NESTA_COLOURS[0],
    strokeWidth=3
).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('counts:Q', title='Number of publications'),
)
fig = pu.configure_plots(fig)
fig

alt.Chart(...)

In [58]:
_df

,year,counts
4,2017.0,1247
5,2018.0,1230
6,2019.0,1455
7,2020.0,1541
8,2021.0,1538
9,2022.0,1425
10,2023.0,1689


In [59]:
# get growth percentage from 2017 to 2023
au.ts_magnitude_growth_(
    ts_df = _df,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,1529.6,18.311292


In [61]:
 _data_exploded_df.query("type == 'Society'").query("topic in @allowed_topics")

,id,text,dataset,topics,year,country_code,topic,type,subtype,name
4,W3097547541,Does Household Income Affect children’s Outcom...,openalex,income,2020.0,[GB],income,Society,Income,Income
32,W3095952743,Associations between high temperatures in preg...,openalex,inequality,2020.0,"[DE, GB, ZA, IE, AU]",inequality,Society,Inequalities,Inequalities
99,W3118554394,Cortisol and socioeconomic status in early chi...,openalex,income,2020.0,"[US, GB]",income,Society,Income,Income
103,W3118554394,Cortisol and socioeconomic status in early chi...,openalex,inequality,2020.0,"[US, GB]",inequality,Society,Inequalities,Inequalities
121,W3092183648,Socioeconomic Factors Account for Variability ...,openalex,income,2020.0,[US],income,Society,Income,Income
...,...,...,...,...,...,...,...,...,...,...
189585,W4390012438,Characterization of parent and youth-reported ...,openalex,income,2023.0,"[US, GB]",income,Society,Income,Income
189586,W4390012438,Characterization of parent and youth-reported ...,openalex,inequality,2023.0,"[US, GB]",inequality,Society,Inequalities,Inequalities
189594,W4363675597,Association between intimate partner violence ...,openalex,income,2023.0,"[RW, KE]",income,Society,Income,Income
189595,W4363675597,Association between intimate partner violence ...,openalex,inequality,2023.0,"[RW, KE]",inequality,Society,Inequalities,Inequalities
